# Meta-analysis of the UCI results

All methods of the UCI tables compared across datasets and splits at once, one mixed model per
metric and architecture. The method is a fixed effect against a baseline method, and the dataset
and the split within it get random intercepts, which absorb that datasets differ in difficulty and
that all methods share the same splits. Details in `utils/meta_analysis.py`.

* **RMSE, CRPS**: Gaussian mixed model on the log metric, shown as % difference to the baseline.
* **NLL**: Gaussian mixed model on the NLL itself (it can be negative), shown as a difference.
* **C(0.9)**: binomial mixed model on whether each test response lies inside the central 90%
  interval (variational Bayes), shown as the coverage for an average dataset and split.

Brackets are 95% intervals. Positive RMSE, CRPS and NLL differences mean worse than the baseline.
The residual variance is shared across datasets, so intervals are wide where one dataset (yacht)
varies a lot between splits.

In [ ]:
import os, warnings
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
warnings.filterwarnings("ignore")      # statsmodels convergence chatter, check `converged` below

import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 20)
from notebooks.utils import meta_analysis as M

df = M.results_frame()                 # one row per (arch, method, dataset, split)
pts = M.coverage_points()              # one row per test point and run, from the coverage cache
df.groupby(["arch", "method"]).size().unstack(0)

## One table per architecture

Baseline: the sticky ZigZag at $w = 0.3$. Change `BASELINE` to compare against another method.

In [ ]:
BASELINE = M.BASELINE                  # e.g. "NUTS" or "Sticky ZigZag (w=0.1)"
tables = {arch: M.summary(df, pts, arch, baseline=BASELINE) for arch in ("small", "medium")}
for arch, t in tables.items():
    print(f"\n=== {arch} network, baseline {BASELINE}")
    print(t.to_string())

## A single model in detail

Estimates, intervals and p-values for one metric. `converged` should be True.

In [ ]:
ARCH, METRIC, LOG = "small", "RMSE", True      # LOG=False for NLL
t = M.fit_lmm(df[df.arch == ARCH], METRIC, log=LOG, baseline=BASELINE)
print(t.attrs)
t.round(3)

In [ ]:
# coverage per method for one architecture, with intervals
M.fit_coverage(pts[pts.arch == ARCH], baseline=BASELINE).round(3)

## LaTeX for the supplement

In [ ]:
for arch, t in tables.items():
    d = 752 if arch == "small" else 5852
    print(M.to_latex(t, caption=(f"Mixed-effects comparison over all UCI datasets and splits for the {arch} network "
                                 f"(Boston $d={d}$), against the {BASELINE}. RMSE and CRPS are percentage differences, "
                                 "NLL a difference, and $C_{0.9}$ the coverage of the central 90\\% interval for an average "
                                 "dataset and split, each with a 95\\% interval."),
                     label=f"tab:supp_meta_{arch}"))